In [1]:
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder
from torchinfo import summary
from torch.utils.data import DataLoader, Subset
import os
import time

In [2]:
if torch.cuda.is_available():
    device = "cuda" # Use NVIDIA GPU 
else:
    device = "cpu"
device

'cpu'

In [3]:
# Veriyi Colab/bilgisayarına indirmesi ve başlatması için:
# data = datasets.Imagenette(root="./data", split="train", download=True, size="320px")

In [4]:
data_dir = "./data/imagenette2-320"

train_dir = os.path.join(data_dir, "train")
val_dir = os.path.join(data_dir, "val")

train_dataset = ImageFolder(root=train_dir, transform=transforms.ToTensor())
val_dataset = ImageFolder(root=val_dir, transform=transforms.ToTensor())

In [5]:
len(train_dataset)

9469

In [6]:
len(val_dataset)

3925

In [7]:
img, label = train_dataset[3031]

In [8]:
label

3

In [9]:
img

tensor([[[1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         [1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         [1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         ...,
         [0.8863, 0.8824, 0.8784,  ..., 0.8902, 0.8941, 0.8980],
         [0.8824, 0.8784, 0.8745,  ..., 0.8863, 0.8902, 0.8941],
         [0.8824, 0.8784, 0.8745,  ..., 0.8824, 0.8863, 0.8902]],

        [[1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         [1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         [1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         ...,
         [0.8824, 0.8784, 0.8745,  ..., 0.8510, 0.8549, 0.8588],
         [0.8784, 0.8745, 0.8706,  ..., 0.8471, 0.8510, 0.8549],
         [0.8784, 0.8745, 0.8706,  ..., 0.8431, 0.8471, 0.8510]],

        [[1.0000, 1.0000, 1.0000,  ..., 0.9412, 0.9412, 0.9412],
         [1.0000, 1.0000, 1.0000,  ..., 0.9412, 0.9412, 0.9412],
         [1.0000, 1.0000, 1.0000,  ..., 0.9412, 0.9412, 0.

In [10]:
img.shape

torch.Size([3, 320, 836])

In [11]:
class Fire(nn.Module):
    def __init__(self, in_channels = 3, s1 = 16, e1 = 64, e3 = 64):
        super().__init__()
        self.squeeze = nn.Conv2d(
            in_channels = in_channels,
            out_channels = s1,
            kernel_size = 1,
            padding = 0
            )
        self.activation = nn.ReLU(inplace = True)

        self.expand1 = nn.Conv2d(in_channels = s1,
                                out_channels = e1,
                                kernel_size = 1,
                                padding = 0)
        
        self.expand3 = nn.Conv2d(in_channels = s1,
                                out_channels = e3,
                                kernel_size = 3,
                                padding = 1)
    def forward(self, x):
        squeeze_out = self.activation(self.squeeze(x))

        out_e1 = self.activation(self.expand1(squeeze_out))
        out_e3 = self.activation(self.expand3(squeeze_out))

        return torch.cat([out_e1, out_e3], dim = 1)

In [12]:
class SqueezeNet(nn.Module):
    def __init__(self, classes = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = 3, 
                                out_channels = 96,
                                kernel_size = 7, 
                                stride = 2, 
                                padding = 3)
        self.relu1 = nn.ReLU(inplace=True)
        self.maxpool1 = nn.MaxPool2d(3, stride = 2)
        
        self.fire2 = Fire(in_channels = 96, s1 = 16, e1 = 64, e3 = 64)
        self.fire3 = Fire(in_channels = 128, s1 = 16, e1 = 64, e3 = 64)
        self.fire4 = Fire(in_channels = 128, s1 = 32, e1 = 128, e3 = 128)
        self.maxpool4 = nn.MaxPool2d(3, stride = 2)

        self.fire5 = Fire(in_channels = 256, s1 = 32, e1 = 128, e3 = 128)
        self.fire6 = Fire(in_channels = 256, s1 = 48, e1 = 192, e3 = 192)
        self.fire7 = Fire(in_channels = 384, s1 = 48, e1 = 192, e3 = 192)
        self.fire8 = Fire(in_channels = 384, s1 = 64, e1 = 256, e3 = 256)
        self.maxpool8 = nn.MaxPool2d(3, stride = 2)

        self.fire9 = Fire(in_channels = 512, s1 = 64, e1 = 256, e3 = 256)

        self.dropout =nn.Dropout(p=0.5)
        
        self.conv10 = nn.Conv2d(in_channels = 512, 
                                out_channels = classes,
                                kernel_size = 1, 
                                stride = 1)
        self.relu10 = nn.ReLU(inplace = True)
        self.avgpool10 = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        first = self.maxpool1(self.relu1(self.conv1(x)))
        blok_1 = self.maxpool4(self.fire4(self.fire3(self.fire2(first))))
        blok_2 = self.maxpool8(self.fire8(self.fire7(self.fire6(self.fire5(blok_1)))))
        blok_3 = self.avgpool10(self.relu10(self.conv10(self.dropout(self.fire9(blok_2)))))  # batch,10,1,1
        return torch.flatten(blok_3, 1)  # (batch,10) 0. indekse dokunma flattena 1den başla

In [13]:
img = img.permute(1,2,0)
img.shape

torch.Size([320, 836, 3])

In [14]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(size = 224, scale=(0.8, 1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [15]:
train_data = ImageFolder(root = "./data/imagenette2-320/train", transform = train_transforms)
val_data = ImageFolder(root = "./data/imagenette2-320/val", transform = val_transforms)

train_loader = DataLoader(train_data, batch_size = 32, shuffle = True)
val_loader = DataLoader(val_data, batch_size = 32, shuffle = False)

In [16]:
model = SqueezeNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model.parameters(), lr = 0.005)

In [17]:
img, label = next(iter(train_loader))

out = model(img)

print(out[:5])
print(label[:5])

tensor([[0.0436, 0.0276, 0.0026, 0.0061, 0.0445, 0.0695, 0.0169, 0.0039, 0.0026,
         0.0072],
        [0.0426, 0.0240, 0.0032, 0.0065, 0.0463, 0.0669, 0.0148, 0.0046, 0.0028,
         0.0074],
        [0.0444, 0.0240, 0.0033, 0.0059, 0.0455, 0.0696, 0.0143, 0.0054, 0.0034,
         0.0057],
        [0.0462, 0.0261, 0.0035, 0.0052, 0.0478, 0.0692, 0.0118, 0.0055, 0.0024,
         0.0049],
        [0.0456, 0.0271, 0.0044, 0.0062, 0.0447, 0.0680, 0.0162, 0.0044, 0.0025,
         0.0085]], grad_fn=<SliceBackward0>)
tensor([9, 9, 0, 9, 3])


In [ ]:
EPOCHS = 10
torch.manual_seed(42)

train_losses = []
val_losses = []
best_val_loss = float("inf")

model = model.to(device)

for epoch in range(EPOCHS):
    epoch_start_time = time.time()
    
    train_epoch_loss = 0
    train_correct = 0
    train_total = 0

    val_epoch_loss = 0
    val_correct = 0 
    val_total = 0

    # === TRAIN PHASING ===
    model.train()
    for img, label in train_loader:
        img = img.to(device)
        label = label.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(img)
        loss = loss_fn(outputs, label)

        loss.backward()
        optimizer.step()

        train_epoch_loss += loss.item() * img.size(0)   # batch boyutunu çarparak ekliyoruz ki ortalamayı alalım
        _, preds = torch.max(outputs, dim=1)
        train_correct += (preds == label).sum().item()
        train_total += label.size(0)

    # === VALIDATION PHASING ===
    model.eval()
    with torch.no_grad():
        for img, label in val_loader:
            img = img.to(device)
            label = label.to(device)

            outputs = model(img)
            loss = loss_fn(outputs, label) 

            val_epoch_loss += loss.item() * img.size(0)
            _, preds = torch.max(outputs, dim=1)
            val_correct += (preds == label).sum().item()
            val_total += label.size(0)

    # === METRIC CALCULATIONS (Hizalama sola çekildi) ===
    epoch_train_loss = train_epoch_loss / len(train_loader.dataset)
    epoch_train_acc = train_correct / train_total
    epoch_val_loss = val_epoch_loss / len(val_loader.dataset)
    epoch_val_acc = val_correct / val_total

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    
    epoch_end_time = time.time()
    elapsed_time = epoch_end_time - epoch_start_time
    mins, secs = divmod(elapsed_time, 60)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Süre: {int(mins)}dk {int(secs)}sn -> "
          f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc*100:.2f}%")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_squeezenet.pth")
        print(f"  --> Yeni en iyi model kaydedildi! (Val Loss: {best_val_loss:.4f})")
    
    print("-" * 40)